# Zajęcia 13 — prezentacje projektów i przegląd produkcyjny

Notebook przygotowuje do obrony: automatyczna lista kontrolna projektu, losowanie scenariusza awarii, protokół demonstracji i formularz wzajemnego przeglądu.

Listę kontrolną uruchamiasz na **swoim** repozytorium — wystarczy podać ścieżkę.

Komórki opisane jako **zadanie** wypełniasz samodzielnie. Komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True`.

## 1. Przygotowanie notebooka

Budujemy też mały projekt przykładowy, żeby lista kontrolna miała na czym działać, zanim wskażesz własne repozytorium. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import hashlib
import re
import sys
import textwrap
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE_DIR = REPO_ROOT / "artifacts" / "zajecia_13"
REPORTS_DIR = BASE_DIR / "reports"
DEMO_PROJECT = BASE_DIR / "projekt_przykladowy"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
for sub in ("src", "tests", "config"):
    (DEMO_PROJECT / sub).mkdir(parents=True, exist_ok=True)


def write(relative_path: str, content: str) -> None:
    path = DEMO_PROJECT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")


write(
    "README.md",
    """
    # Projekt przykladowy

    ## Uruchomienie

    ```bash
    pip install -r requirements.txt
    make all
    ```

    Dane pobiera skrypt `src/download.py` i weryfikuje sume kontrolna z `data_manifest.json`.
    """,
)
write("requirements.txt", "pandas==3.0.5\nscikit-learn==1.9.0\n")
write("Makefile", "all:\n\tpython src/cli.py all\n")
write(".gitignore", "data/\nartifacts/\n.venv/\n.env\n__pycache__/\n")
write("config/config.yaml", "seed: 42\ntest_size: 0.2\n")
write("data_manifest.json", '{"name": "demo", "version": "1.0.0", "sha256": "abc123"}\n')
write("src/cli.py", "# interfejs wiersza polecen\n")
write("tests/test_features.py", "def test_placeholder():\n    assert True\n")
write("MODEL_CARD.md", "# Karta modelu\n\nPrzeznaczenie, dane, wyniki, ograniczenia.\n")

# Artefakty, ktore doszly w trakcie semestru - projekt przykladowy ma je komplet,
# zeby bylo widac, jak wyglada stan docelowy.
write(".python-version", "3.12\n")
write("uv.lock", "# blokada zaleznosci wygenerowana przez uv\n")
write("ruff.toml", "line-length = 100\n")
write(".pre-commit-config.yaml", "repos:\n  - repo: https://github.com/astral-sh/ruff-pre-commit\n")
write("dvc.yaml", "stages:\n  train:\n    cmd: python src/cli.py train\n")
write("dvc.lock", "schema: '2.0'\n")
write("DATASET_CARD.md", "# Metryczka zbioru\n\nZrodlo, licencja, ograniczenia.\n")
write(
    "Dockerfile",
    "FROM python:3.12-slim AS builder\nFROM python:3.12-slim AS runtime\n"
    "RUN useradd --uid 10001 appuser\nUSER appuser\n",
)
write(".dockerignore", "data/\nartifacts/\n.venv/\n")
write(
    ".github/workflows/ci.yml",
    "name: ci\njobs:\n  quality:\n    steps:\n      - run: pytest\n      - run: python ci/scan_dependencies.py\n"
    "      - uses: gitleaks/gitleaks-action@v2\n",
)
write(
    ".github/workflows/cd.yml",
    "name: cd\njobs:\n  deploy-production:\n    environment: production\n    steps:\n      - run: docker compose up -d\n",
)
write("docs/rollback_runbook.md", "# Procedura wycofania\n\nWarunek, role, kroki, potwierdzenie.\n")
write("docs/threat_model.md", "# Model zagrozen\n\nRyzyka i decyzje.\n")
write("docs/monitoring_report.md", "# Raport monitoringu\n\nSLI, SLO, alerty.\n")

print(f"Projekt przykładowy: {DEMO_PROJECT.relative_to(REPO_ROOT)}")

%load_ext skip_kernel_extension

## 2. Automatyczna lista kontrolna

Lista sprawdza to, co da się sprawdzić maszynowo. Nie zastępuje przeglądu — odsiewa braki, o które szkoda tracić czas na obronie.

**Zadanie:** napisz `check_project(path)` zwracającą ramkę z kolumnami `wymaganie`, `spelnione`, `szczegol`. Sprawdź co najmniej: `README.md`, przypięte zależności, `.gitignore` wykluczający dane i sekrety, katalog testów, konfigurację poza kodem, manifest danych, kartę modelu, jedną komendę uruchomieniową oraz brak sekretów w plikach tekstowych.

In [ ]:
SECRET_PATTERNS = {
    "klucz dostepu do chmury": r"AKIA[0-9A-Z]{16}",
    "klucz prywatny": r"-----BEGIN [A-Z ]*PRIVATE KEY-----",
}

# Zadanie: napisz automatyczną listę kontrolną projektu.


In [ ]:
%%skip True
TEXT_SUFFIXES = {".py", ".txt", ".md", ".yaml", ".yml", ".json", ".cfg", ".ini", ""}
SKIP_DIRS = {".git", ".venv", "__pycache__", "artifacts", "data"}


def find_secrets(path: Path) -> list:
    findings = []
    for item in path.rglob("*"):
        if not item.is_file() or item.suffix not in TEXT_SUFFIXES:
            continue
        if set(item.relative_to(path).parts) & SKIP_DIRS:
            continue
        try:
            content = item.read_text(encoding="utf-8")
        except (UnicodeDecodeError, OSError):
            continue
        for label, pattern in SECRET_PATTERNS.items():
            if re.search(pattern, content):
                findings.append(f"{item.relative_to(path).as_posix()} ({label})")
    return findings


def check_project(path) -> pd.DataFrame:
    path = Path(path)
    checks = []

    def add(name: str, passed: bool, detail: str = "", obszar: str = "podstawy") -> None:
        checks.append(
            {
                "obszar": obszar,
                "wymaganie": name,
                "spelnione": bool(passed),
                "szczegol": detail,
            }
        )

    readme = path / "README.md"
    add("README.md istnieje", readme.exists())
    if readme.exists():
        text = readme.read_text(encoding="utf-8")
        has_command = "```" in text
        add("README zawiera instrukcje uruchomienia", has_command, "" if has_command else "brak bloku z komenda")

    requirements = path / "requirements.txt"
    pyproject = path / "pyproject.toml"
    add("zadeklarowane zaleznosci", requirements.exists() or pyproject.exists())
    if requirements.exists():
        lines = [
            line
            for line in requirements.read_text(encoding="utf-8").splitlines()
            if line.strip() and not line.strip().startswith("#")
        ]
        add("wszystkie zaleznosci przypiete", all("==" in line for line in lines), f"{len(lines)} pozycji")

    gitignore = path / ".gitignore"
    add(".gitignore istnieje", gitignore.exists())
    if gitignore.exists():
        ignored = gitignore.read_text(encoding="utf-8")
        add("gitignore wyklucza dane i sekrety", "data/" in ignored and ".env" in ignored)

    add("katalog testow", (path / "tests").is_dir())
    add(
        "konfiguracja poza kodem",
        (path / "config").is_dir() or any(path.glob("*.yaml")) or any(path.glob("*.toml")),
    )
    add(
        "manifest danych",
        any(path.glob("*manifest*")) or (path / "dvc.yaml").exists() or (path / "data.dvc").exists(),
    )
    add("karta modelu", any(path.glob("*MODEL_CARD*")) or any(path.glob("docs/model_card*")))
    add(
        "jedna komenda uruchomieniowa",
        (path / "Makefile").exists() or (path / "justfile").exists() or (path / "tasks.py").exists(),
    )

    # --- srodowisko (zajecia 2) ---
    add("blokada zaleznosci", (path / "uv.lock").exists() or requirements.exists(), obszar="srodowisko")
    add("zapisana wersja Pythona", (path / ".python-version").exists(), obszar="srodowisko")
    ruff_config = (path / "ruff.toml").exists() or (
        pyproject.exists() and "[tool.ruff" in pyproject.read_text(encoding="utf-8")
    )
    add("konfiguracja kontroli stylu", ruff_config, obszar="srodowisko")
    add("hak przed commitem", (path / ".pre-commit-config.yaml").exists(), obszar="srodowisko")

    # --- dane (zajecia 3 i 4) ---
    add(
        "dane pod kontrola wersji",
        (path / "dvc.yaml").exists() or any(path.rglob("*.dvc")),
        obszar="dane",
    )
    add("zapisany stan potoku (dvc.lock)", (path / "dvc.lock").exists(), obszar="dane")
    add(
        "metryczka zbioru",
        any(path.glob("*DATASET_CARD*")) or any(path.glob("docs/dataset_card*")),
        obszar="dane",
    )

    # --- CI i lancuch dostaw (zajecia 6 i 9) ---
    workflows = list((path / ".github" / "workflows").glob("*.y*ml"))
    workflow_text = "\n".join(item.read_text(encoding="utf-8") for item in workflows)
    add("workflow ciaglej integracji", bool(workflows), f"{len(workflows)} plikow", "CI")
    add("CI uruchamia testy", "pytest" in workflow_text, obszar="CI")
    add(
        "CI skanuje zaleznosci i sekrety",
        ("pip_audit" in workflow_text or "pip-audit" in workflow_text or "scan_dependencies" in workflow_text)
        and ("gitleaks" in workflow_text or "scan_secrets" in workflow_text),
        obszar="CI",
    )

    # --- kontener (zajecia 9) ---
    dockerfile = path / "Dockerfile"
    add("Dockerfile", dockerfile.exists(), obszar="kontener")
    if dockerfile.exists():
        content = dockerfile.read_text(encoding="utf-8")
        from_lines = [line for line in content.splitlines() if line.strip().upper().startswith("FROM")]
        add("budowanie wieloetapowe", len(from_lines) >= 2, obszar="kontener")
        add(
            "obraz bazowy przypiety",
            bool(from_lines) and all(":" in line and ":latest" not in line for line in from_lines),
            obszar="kontener",
        )
        add("proces bez uprawnien administratora", "USER " in content, obszar="kontener")
    add(".dockerignore", (path / ".dockerignore").exists(), obszar="kontener")

    # --- wdrozenie i obserwowalnosc (zajecia 10 i 11) ---
    add(
        "opis wdrozenia",
        "deploy" in workflow_text or any(path.rglob("compose*.y*ml")),
        obszar="wdrozenie",
    )
    add(
        "procedura wycofania",
        any(path.rglob("*rollback*")) or any(path.rglob("*runbook*")),
        obszar="wdrozenie",
    )
    add(
        "raport monitoringu albo endpoint metryk",
        any(path.rglob("*monitoring*")) or "generate_latest" in workflow_text
        or any("generate_latest" in item.read_text(encoding="utf-8", errors="ignore")
               for item in path.rglob("*.py")),
        obszar="obserwowalnosc",
    )

    # --- governance (zajecia 7, 10 i 12) ---
    add("model zagrozen", any(path.rglob("*threat*")), obszar="governance")

    secrets = find_secrets(path)
    add("brak sekretow w plikach tekstowych", not secrets, ", ".join(secrets), "governance")

    return pd.DataFrame(checks)


checklist = check_project(DEMO_PROJECT)
checklist.to_csv(REPORTS_DIR / "project_checklist.csv", index=False)
display(checklist)

print(f"Spełnione: {int(checklist['spelnione'].sum())} z {len(checklist)}\n")
display(
    checklist.groupby("obszar").agg(
        wymagan=("spelnione", "size"), spelnionych=("spelnione", "sum")
    )
)

## 3. Czego lista kontrolna nie sprawdzi

Automat potrafi stwierdzić, że plik **istnieje**. Nie potrafi stwierdzić, że jest **prawdziwy**. `Dockerfile` może istnieć i nie budować się; workflow może istnieć i nigdy nie przejść; runbook może opisywać procedurę, której nikt nie wykonał.

Dlatego lista kontrolna dzieli się na dwie części: automatyczną, którą właśnie uruchomiliśmy, i **dowody**, których automat nie zastąpi.

Część, którą sprawdza się na GitHubie, w przeglądarce:

| Dowód | Gdzie szukać | Co ma być widać |
|---|---|---|
| CI jest bramką | Settings → Branches | reguła ochrony `main` z wymaganym przebiegiem |
| CI działa | zakładka Actions | ostatni przebieg na `main` zakończony powodzeniem |
| bramka zadziałała | historia zgłoszeń zmian | zgłoszenie zablokowane przez czerwony przebieg |
| obraz powstał | zakładka Packages | opublikowany obraz i jego digest |
| wdrożenie kontrolowane | Actions → uruchomienie `cd` | zadanie produkcyjne czekające na zatwierdzenie |
| wycofanie przetestowane | sprawozdanie | zmierzony czas powrotu do poprzedniego digestu |

Ostatni wiersz jest najważniejszy. Procedura wycofania, której nigdy nie wykonano, jest wypracowaniem, nie procedurą.

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import shutil
import subprocess

EVIDENCE = [
    ("ochrona galezi main z wymaganym przebiegiem", "Settings -> Branches"),
    ("ostatni przebieg CI na main zakonczony powodzeniem", "Actions"),
    ("zgloszenie zmiany zablokowane przez czerwony przebieg", "Pull requests"),
    ("obraz opublikowany w rejestrze wraz z digestem", "Packages"),
    ("wdrozenie produkcyjne za bramka zatwierdzenia", "Actions -> cd"),
    ("wycofanie wykonane i zmierzone", "sprawozdanie"),
]


def check_git(path: Path) -> pd.DataFrame:
    """Sprawdza to, co da sie ustalic lokalnie z repozytorium."""
    rows = []

    def add(wymaganie: str, spelnione: bool, szczegol: str = "") -> None:
        rows.append(
            {"obszar": "git", "wymaganie": wymaganie, "spelnione": bool(spelnione), "szczegol": szczegol}
        )

    if shutil.which("git") is None:
        add("polecenie git dostepne", False, "brak git w PATH")
        return pd.DataFrame(rows)

    def git(*arguments) -> subprocess.CompletedProcess:
        return subprocess.run(["git", *arguments], cwd=path, capture_output=True, text=True)

    # Uwaga: `--is-inside-work-tree` odpowiada twierdzaco takze dla katalogu
    # zagniezdzonego w cudzym repozytorium. Sprawdzamy wiec korzen.
    toplevel = git("rev-parse", "--show-toplevel").stdout.strip()
    is_repo = bool(toplevel) and Path(toplevel).resolve() == path.resolve()
    add(
        "katalog jest korzeniem repozytorium Git",
        is_repo,
        "" if is_repo else f"korzen wykryty w {toplevel or 'brak'}",
    )
    if not is_repo:
        return pd.DataFrame(rows)

    remotes = git("remote", "-v").stdout.strip()
    add("skonfigurowane repozytorium zdalne", bool(remotes), remotes.splitlines()[0][:60] if remotes else "")

    tracked_data = git("ls-files", "data").stdout.strip().splitlines()
    leaked = [name for name in tracked_data if not name.endswith((".dvc", ".gitignore", ".md"))]
    add("dane nie sa sledzone przez Git", not leaked, ", ".join(leaked[:3]))

    commits = git("rev-list", "--count", "HEAD").stdout.strip()
    add("historia zawiera wiecej niz jeden commit", commits.isdigit() and int(commits) > 1, f"{commits} commitow")

    return pd.DataFrame(rows)


display(check_git(DEMO_PROJECT))

evidence = pd.DataFrame(
    [{"dowod": name, "gdzie sprawdzic": where, "potwierdzone": ""} for name, where in EVIDENCE]
)
evidence.to_csv(REPORTS_DIR / "evidence_checklist.csv", index=False)
print(f"\nLista dowodów do wypełnienia ręcznie: {(REPORTS_DIR / 'evidence_checklist.csv').name}")
display(evidence)

## 4. Lista kontrolna na własnym projekcie

Podmień ścieżkę na katalog swojego repozytorium i uruchom ponownie. Wynik zapisz i napraw to, co się da przed obroną.

Ta komórka jest gotowa — zmień wyłącznie ścieżkę.

In [ ]:
MOJ_PROJEKT = DEMO_PROJECT  # <- podmien na sciezke do wlasnego repozytorium

result = pd.concat([check_project(Path(MOJ_PROJEKT)), check_git(Path(MOJ_PROJEKT))], ignore_index=True)
result.to_csv(REPORTS_DIR / "project_checklist.csv", index=False)

print(f"Spełnione: {int(result['spelnione'].sum())} z {len(result)}\n")

failed = result[~result["spelnione"]]
if failed.empty:
    print("Wszystkie automatyczne wymagania spełnione.")
else:
    print("Do naprawy przed obroną:")
    display(failed[["obszar", "wymaganie", "szczegol"]])

## 5. Losowanie scenariusza awarii

Losowanie jest jawne i powtarzalne: scenariusz wynika z identyfikatora studenta, więc ten sam identyfikator zawsze daje ten sam wynik i nikt nie może zarzucić stronniczości.

**Zadanie:** napisz `draw_scenario(identifier)` wyznaczającą scenariusz na podstawie skrótu identyfikatora.

In [ ]:
SCENARIOS = [
    "Plik z danymi wejsciowymi zostal podmieniony i ma inna liczbe kolumn.",
    "Jedna kolumna zmienila jednostke - wartosci sa dziesieciokrotnie wieksze.",
    "Model ponizej progu jakosci zostal promowany przez pomylke.",
    "Rejestr modeli jest niedostepny w momencie startu uslugi.",
    "Sekret uzywany przez usluge wygasl.",
    "Opoznienie p99 wzroslo pieciokrotnie bez zmiany w kodzie.",
    "Rozklad danych wejsciowych przesunal sie, ale metryka jakosci nie spadla.",
    "Etykiety nie naplynely przez dwa okresy raportowe.",
    "Konsument zglasza, ze odpowiedz ma inny format niz w dokumentacji.",
    "Ponowny trening wytrenowal model na danych z okresu awarii.",
]

# Zadanie: napisz powtarzalne losowanie scenariusza.


In [ ]:
%%skip True
def draw_scenario(identifier: str) -> dict:
    digest = hashlib.sha256(identifier.encode("utf-8")).hexdigest()
    index = int(digest[:8], 16) % len(SCENARIOS)
    return {"identyfikator": identifier, "numer": index + 1, "scenariusz": SCENARIOS[index]}


examples = [
    draw_scenario(name) for name in ("student_01", "student_02", "student_03", "student_01")
]
display(pd.DataFrame(examples))
print("Ten sam identyfikator zawsze daje ten sam scenariusz — losowanie jest powtarzalne.")

## 6. Protokół demonstracji

Protokół jest dowodem, że demonstracja się odbyła i jak przebiegła. Wypełniasz go w trakcie obrony.

**Zadanie:** wygeneruj `reports/demo_protocol.md` zawierający: sposób demonstracji, wynik listy kontrolnej, wylosowany scenariusz oraz trzy pola do wypełnienia — wykrycie, diagnoza, reakcja.

In [ ]:
STUDENT = "student_01"
PROJECT_TAG = "v1.0"

# Zadanie: wygeneruj protokół demonstracji.


In [ ]:
%%skip True
drawn = draw_scenario(STUDENT)
passed = int(result["spelnione"].sum())
missing = ", ".join(result.loc[~result["spelnione"], "wymaganie"]) or "brak"

protocol = f"""# Protokol demonstracji

- Student: {STUDENT}
- Wersja projektu: `{PROJECT_TAG}`
- Data: {date.today().isoformat()}

## Lista kontrolna

Spelnione automatyczne wymagania: {passed} z {len(result)}.

Niespelnione: {missing}

## Demonstracja powtarzalnosci

- [ ] swiezy klon i uruchomienie jedna komenda
- [ ] przebieg CI na czystym runnerze
- [ ] uruchomienie kontenera zbudowanego z repozytorium

Wynik demonstracji: ..................................................

## Scenariusz awarii

Numer {drawn['numer']}: {drawn['scenariusz']}

### Wykrycie

Po czym poznalbym, ze to sie dzieje: ..................................

### Diagnoza

Ktore artefakty sprawdzam i w jakiej kolejnosci: ......................

### Reakcja

Co robie natychmiast, a co po ustaleniu przyczyny: ....................

## Pytanie prowadzacego

Pytanie: .............................................................

Odpowiedz: ...........................................................

## Uwagi

.......................................................................
"""

(REPORTS_DIR / "demo_protocol.md").write_text(protocol, encoding="utf-8")
print(protocol)

## 7. Formularz wzajemnego przeglądu

Przegląd ma być konkretny. Zamiast „dokumentacja mogłaby być lepsza” — „w README brakuje informacji, skąd pobrać dane”.

Ta komórka jest gotowa — uruchom ją bez zmian, a wypełniony formularz przekaż autorowi ocenianego projektu.

In [ ]:
review = """# Wzajemny przeglad projektu

- Oceniajacy: ..........................
- Oceniany projekt: ....................
- Wersja (tag): ........................

## 1. Odtwarzalnosc

Czy udalo sie odtworzyc wynik wylacznie na podstawie README, bez pytania autora?

- [ ] tak, za pierwszym razem
- [ ] tak, po jednym pytaniu
- [ ] nie

Co dokladnie zablokowalo odtworzenie: .................................

## 2. Identyfikowalnosc

Czy da sie wskazac, z jakich danych i z jakiego kodu powstal wdrozony model?

Sciezka, ktora przeszedlem: ...........................................

## 3. Testy

Przed czym realnie chronia testy w tym projekcie? Podaj jeden konkretny przyklad.

.......................................................................

## 4. Higiena repozytorium

- [ ] brak sekretow w historii
- [ ] brak duzych plikow binarnych
- [ ] dane i artefakty wykluczone z Git

Uwagi: ................................................................

## 5. Podsumowanie

Jedna rzecz zrobiona dobrze: ..........................................

Jedna rzecz do poprawy (konkretnie): ..................................
"""

(REPORTS_DIR / "peer_review.md").write_text(review, encoding="utf-8")
print(review)

## 8. Podsumowanie przedmiotu

Wracamy do pytania z pierwszych zajęć: *czy model o accuracy 0,92 jest gotowy do wdrożenia?*

Przez trzynaście spotkań dołożyliśmy do modelu wszystko, czego brakowało do odpowiedzi „tak”:

| Spotkanie | Co przestało być niewiadomą |
|---:|---|
| 2 | jak uruchomić projekt na innym komputerze |
| 3 | z jakich danych powstał i czy wolno ich używać |
| 4 | czy trening i predykcja liczą cechy tak samo |
| 5 | dlaczego wybrano ten model, a nie inny |
| 6 | czy zmiana w kodzie czegoś nie zepsuła |
| 7 | która wersja modelu jest używana i skąd pochodzi |
| 8 | jak wywołać model spoza notebooka |
| 9 | czy działa poza komputerem autora |
| 10 | jak bezpiecznie zmienić wersję i jak ją wycofać |
| 11 | co się dzieje po wdrożeniu |
| 12 | co zrobić, gdy świat się zmieni |
